# Notebook 2: Kernels and Thread Hierarchy

In this notebook, you will learn:

- The CUDA thread hierarchy in detail (grids, blocks, threads)
- Built-in variables: `threadIdx`, `blockIdx`, `blockDim`, `gridDim`
- How to compute global thread IDs
- Grid-stride loops for handling arbitrary data sizes
- Choosing block and grid dimensions

In [ ]:
%load_ext nvcc4jupyter

## 2.1 The Thread Hierarchy

CUDA organizes threads in a two-level hierarchy:

```
Grid (one kernel launch)
├── Block 0
│   ├── Thread 0
│   ├── Thread 1
│   ├── ...
│   └── Thread N-1
├── Block 1
│   ├── Thread 0
│   ├── Thread 1
│   ├── ...
│   └── Thread N-1
├── ...
└── Block M-1
    ├── Thread 0
    └── ...
```

**Why two levels?**
- Threads within a block can **synchronize** and **share fast memory**
- Blocks are independent — the GPU can schedule them in any order
- This makes CUDA programs scalable across GPUs with different SM counts

## 2.2 Built-in Variables

Every thread has access to these built-in variables:

| Variable | Type | Meaning |
|----------|------|---------|
| `threadIdx.x` | `uint3` | Thread index within its block |
| `blockIdx.x` | `uint3` | Block index within the grid |
| `blockDim.x` | `dim3` | Number of threads per block |
| `gridDim.x` | `dim3` | Number of blocks in the grid |

Each has `.x`, `.y`, `.z` components for up to 3D indexing.

In [ ]:
%%cuda
#include <cstdio>

__global__ void show_ids() {
    printf("blockIdx=(%d,%d) threadIdx=(%d,%d) | blockDim=(%d,%d) gridDim=(%d,%d)\n",
           blockIdx.x, blockIdx.y,
           threadIdx.x, threadIdx.y,
           blockDim.x, blockDim.y,
           gridDim.x, gridDim.y);
}

int main() {
    // Launch a 2x2 grid of 3x2 blocks = 24 threads total
    dim3 grid(2, 2);    // 2 blocks in x, 2 in y
    dim3 block(3, 2);   // 3 threads in x, 2 in y per block

    show_ids<<<grid, block>>>();
    cudaDeviceSynchronize();
    return 0;
}

## 2.3 Computing the Global Thread ID

The most fundamental pattern in CUDA: mapping each thread to a unique data element.

### 1D case (most common)

```
Block 0          Block 1          Block 2
[T0 T1 T2 T3]   [T0 T1 T2 T3]   [T0 T1 T2 T3]
 0  1  2  3      4  5  6  7      8  9  10 11     ← global ID
```

Formula: **`global_id = blockIdx.x * blockDim.x + threadIdx.x`**

In [ ]:
%%cuda
#include <cstdio>

__global__ void global_id_demo() {
    int gid = blockIdx.x * blockDim.x + threadIdx.x;
    printf("Block %d, Thread %d → Global ID %d\n",
           blockIdx.x, threadIdx.x, gid);
}

int main() {
    // 3 blocks of 4 threads = 12 threads (IDs 0..11)
    global_id_demo<<<3, 4>>>();
    cudaDeviceSynchronize();
    return 0;
}

### Total number of threads

```cpp
int total_threads = gridDim.x * blockDim.x;  // 1D
```

This is important: we almost never launch exactly the right number of threads for our data. We'll need **bounds checking**.

## 2.4 Handling Arbitrary Data Sizes — Bounds Checking

Problem: Your data has 1000 elements but your grid launches 1024 threads (e.g., 4 blocks × 256 threads). The extra 24 threads must do nothing.

**Always check bounds:**
```cpp
__global__ void process(float* data, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {  // ← bounds check
        data[idx] = data[idx] * 2.0f;
    }
}
```

In [ ]:
%%cuda
#include <cstdio>

__global__ void bounded_kernel(int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        printf("Thread %d: processing element %d ✓\n", idx, idx);
    } else {
        printf("Thread %d: out of bounds (N=%d), skipping\n", idx, N);
    }
}

int main() {
    int N = 10;
    int block_size = 4;
    // Ceiling division to get enough blocks
    int grid_size = (N + block_size - 1) / block_size;  // = 3

    printf("N=%d, grid=%d, block=%d, total threads=%d\n",
           N, grid_size, block_size, grid_size * block_size);

    bounded_kernel<<<grid_size, block_size>>>(N);
    cudaDeviceSynchronize();
    return 0;
}

### The Ceiling Division Pattern

You'll use this constantly:

```cpp
int grid_size = (N + block_size - 1) / block_size;
```

This ensures we have **at least enough** blocks to cover all N elements. Examples:
- N=10, block=4 → grid = (10+3)/4 = 3 → 12 threads (2 extra, handled by bounds check)
- N=16, block=4 → grid = (16+3)/4 = 4 → 16 threads (perfect fit)
- N=1000000, block=256 → grid = 3907 → 999,936+64 threads

## 2.5 Grid-Stride Loops

What if your data has 100 million elements but you can only launch ~100K threads? Use a **grid-stride loop**: each thread processes multiple elements by striding across the data.

```
Data:    [0  1  2  3  4  5  6  7  8  9  10  11 ...]
Thread 0: 0        3        6         9
Thread 1:    1        4        7         10
Thread 2:       2        5        8         11
          └ stride = total threads (gridDim.x * blockDim.x) ┘
```

In [ ]:
%%cuda
#include <cstdio>

__global__ void grid_stride_demo(int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = gridDim.x * blockDim.x;

    // Each thread loops over its assigned elements
    for (int i = idx; i < N; i += stride) {
        printf("Thread %d processes element %d\n", idx, i);
    }
}

int main() {
    int N = 20;
    // Only launch 6 threads (2 blocks of 3) to handle 20 elements
    grid_stride_demo<<<2, 3>>>(N);
    cudaDeviceSynchronize();
    return 0;
}

### Why grid-stride loops are best practice:

1. **Works for any N** — no need to calculate the perfect grid size
2. **Reuses threads** — amortizes launch overhead across more work
3. **Flexible** — same kernel works whether N is 100 or 100 million
4. **Debugging** — launch with `<<<1, 1>>>` to get serial execution for debugging

## 2.6 Choosing Block Size

Rules of thumb:

1. **Must be a multiple of 32** (warp size). Common: 128, 256, 512.
2. **128 or 256** is a good default for most kernels.
3. **Max 1024** threads per block (hardware limit).
4. More threads per block → more chances for latency hiding, but fewer blocks per SM.

For beginners: **use 256 as your default block size.**

In [ ]:
%%cuda
#include <cstdio>

// A practical kernel using standard patterns
__global__ void double_elements(float* data, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = gridDim.x * blockDim.x;

    for (int i = idx; i < N; i += stride) {
        data[i] *= 2.0f;
    }
}

int main() {
    const int N = 1000;
    const int BLOCK_SIZE = 256;
    int grid_size = (N + BLOCK_SIZE - 1) / BLOCK_SIZE;

    printf("N=%d, block_size=%d, grid_size=%d, total_threads=%d\n",
           N, BLOCK_SIZE, grid_size, grid_size * BLOCK_SIZE);

    // Allocate, fill, process, and verify (details in next notebooks)
    float* d_data;
    cudaMalloc(&d_data, N * sizeof(float));

    // Fill with ones on host, copy to device
    float h_data[1000];
    for (int i = 0; i < N; i++) h_data[i] = 1.0f;
    cudaMemcpy(d_data, h_data, N * sizeof(float), cudaMemcpyHostToDevice);

    // Run kernel
    double_elements<<<grid_size, BLOCK_SIZE>>>(d_data, N);

    // Copy back and verify
    cudaMemcpy(h_data, d_data, N * sizeof(float), cudaMemcpyDeviceToHost);
    printf("First 5 elements after doubling: %.1f %.1f %.1f %.1f %.1f\n",
           h_data[0], h_data[1], h_data[2], h_data[3], h_data[4]);

    cudaFree(d_data);
    return 0;
}

## 2.7 Warp Execution

Under the hood, the GPU groups threads into **warps** of 32 threads. All threads in a warp execute the same instruction at the same time (SIMT — Single Instruction, Multiple Threads).

```
Block (128 threads)
├── Warp 0: threads  0-31
├── Warp 1: threads 32-63
├── Warp 2: threads 64-95
└── Warp 3: threads 96-127
```

**Warp divergence:** If threads in the same warp take different branches in an `if` statement, both paths execute serially (the other threads are masked off). This costs performance.

```cpp
// BAD: threads in same warp diverge
if (threadIdx.x % 2 == 0) {
    // Even threads do this
} else {
    // Odd threads do this (both paths execute for entire warp)
}

// BETTER: divergence at warp boundary
if (threadIdx.x / 32 == 0) {
    // First warp does this
} else {
    // Other warps do this (no divergence within a warp)
}
```

In [ ]:
%%cuda
#include <cstdio>

__global__ void warp_demo() {
    int tid = threadIdx.x;
    int warp_id = tid / 32;
    int lane_id = tid % 32;  // position within the warp

    // Only print for first lane of each warp to reduce output
    if (lane_id == 0) {
        printf("Warp %d: threads %d-%d\n", warp_id, tid, tid + 31);
    }
}

int main() {
    // 128 threads = 4 warps
    warp_demo<<<1, 128>>>();
    cudaDeviceSynchronize();
    return 0;
}

## 2.8 Synchronization with `__syncthreads()`

Threads within the same block can synchronize using `__syncthreads()`. This creates a **barrier**: all threads in the block must reach this point before any can continue.

**Important:** `__syncthreads()` only synchronizes within a block, not across blocks!

In [ ]:
%%cuda
#include <cstdio>

__global__ void sync_demo() {
    int tid = threadIdx.x;

    printf("Thread %d: Phase 1\n", tid);

    __syncthreads();  // All threads must reach here before continuing

    printf("Thread %d: Phase 2 (all threads finished phase 1)\n", tid);
}

int main() {
    sync_demo<<<1, 4>>>();
    cudaDeviceSynchronize();
    return 0;
}

**Warning:** Never put `__syncthreads()` inside a conditional that not all threads reach:

```cpp
// DANGEROUS — deadlock if some threads skip the sync!
if (threadIdx.x < 16) {
    __syncthreads();  // Threads ≥16 never reach this → deadlock
}
```

## 2.9 Key Takeaways

1. **Global ID formula:** `idx = blockIdx.x * blockDim.x + threadIdx.x`
2. **Ceiling division** for grid size: `(N + block_size - 1) / block_size`
3. **Always bounds-check:** `if (idx < N)` in every kernel
4. **Grid-stride loops** are the most flexible pattern — use them by default
5. **Block size:** Use multiples of 32, default to 256
6. **Warps** of 32 execute in lockstep — avoid intra-warp divergence
7. **`__syncthreads()`** synchronizes within a block only

## Exercises

**Exercise 2.1:** Write a kernel using a grid-stride loop that sets `data[i] = i * i` for an array of 10,000 elements. Launch with only 2 blocks of 256 threads. Verify the last element is correct.

**Exercise 2.2:** Write a kernel that prints its warp ID and lane ID. Launch with 2 blocks of 64 threads. How many warps total?

**Exercise 2.3:** Modify the grid size calculation to handle N=0 without launching any work. What happens if you pass negative N?

In [ ]:
%%cuda
// Exercise 2.1 — Your code here
#include <cstdio>

__global__ void fill_squares(int* data, int N) {
    // TODO: Grid-stride loop setting data[i] = i * i
}

int main() {
    const int N = 10000;
    int *d_data, *h_data;

    h_data = (int*)malloc(N * sizeof(int));
    cudaMalloc(&d_data, N * sizeof(int));

    // TODO: Launch with 2 blocks of 256 threads

    cudaMemcpy(h_data, d_data, N * sizeof(int), cudaMemcpyDeviceToHost);
    printf("Last element: data[9999] = %d (expected %d)\n",
           h_data[9999], 9999 * 9999);

    free(h_data);
    cudaFree(d_data);
    return 0;
}

---
**Next:** [Notebook 3 — Memory Management](03_Memory_Management.ipynb) — Moving data between CPU and GPU.